# HoloMine Task 2 — Kombinasi HuggingFace yang DirekomendasikanSemua kode ada di satu file: **`holomine_solution.py`**.**Setting notebook:** Accelerator `GPU T4 x2` (atau P100), Internet `ON`, lalu Add Data → dataset kompetisi.| Tahap | Model | Peran ||---|---|---|| 1 | TF-IDF ridge/SVR, kNN kosinus, LightGBM | dasar tanpa GPU, ~10 menit || 2 | *(blend sementara)* | submission valid sedini mungkin || 3 | `Alibaba-NLP/gte-modernbert-base` (beku) | embedding murah, sudut pandang berbeda || 4 | `microsoft/deberta-v3-base` (fine-tune) | pemahaman semantik terkuat || 5 | `answerdotai/ModernBERT-base` (fine-tune) | arsitektur & tokenizer berbeda || 6 | greedy blend + stacking + kalibrasi | menggabungkan semuanya |Semua model dilatih pada `log(listPrice)` dengan loss L1/Huber lalu di-`exp()`,karena MAE diminimalkan oleh **median bersyarat**.

In [ ]:
!git clone --depth 1 https://github.com/AffrizaWildanFauzan/hology_tasks2.git /kaggle/working/repo!pip install -q -U "transformers>=4.48" "sentence-transformers>=3.0" sentencepiece lightgbm

In [ ]:
import osos.environ["HOLOMINE_ARTIFACTS"]   = "/kaggle/working/artifacts"os.environ["HOLOMINE_SUBMISSIONS"] = "/kaggle/working"SOLUSI = "/kaggle/working/repo/holomine_solution.py"# Cek GPU + lihat rencana sebelum membakar kuota!nvidia-smi --query-gpu=name,memory.total --format=csv!python {SOLUSI} --dry-run

## Jalankan semuanya`run_all.py` melewati tahap yang artefaknya sudah ada, jadi kalau sesi Kaggleterputus cukup jalankan ulang sel ini — pekerjaan yang sudah selesai tidak diulang.Perkiraan waktu di T4: CPU ~10 menit, embedding ~10 menit,fine-tune ~45-60 menit **per model** untuk 5 fold.Kalau kuota mepet, pakai `--folds 0,1,2` dulu.

In [ ]:
!python {SOLUSI} --tier core --out submission.csv

## Kalau masih ada kuota GPU`--tier extra` menambah `deberta-v3-large`, `ModernBERT-large`, `bge`, `e5`, dan `qwen`.Model yang sudah dilatih otomatis dilewati, jadi ini hanya melatih yang baru.

In [ ]:
# !python {SOLUSI} --tier extra --out submission.csv

## Periksa hasil sebelum submit

In [ ]:
import pandas as pd, numpy as npsub = pd.read_csv("/kaggle/working/submission.csv")ss  = pd.read_csv([f"/kaggle/input/{d}/sample_submission.csv" for d in os.listdir("/kaggle/input")                   if os.path.exists(f"/kaggle/input/{d}/sample_submission.csv")][0])assert list(sub.columns) == ["id", "listPrice"]assert (sub.id.values == ss.id.values).all(), "urutan id harus sama dengan sample_submission"assert sub.listPrice.notna().all() and (sub.listPrice > 0).all()print(sub.shape)print(sub.head())print("kuantil prediksi:", np.percentile(sub.listPrice, [1, 25, 50, 75, 99]).round(0))